In [ ]:
# === ARC_ATLAS v4: train on hires (50% of full dataset, from 75/25 hires/lores scheme) ===
from pathlib import Path
import importlib.util, os, sys, gc, time, traceback, subprocess, shlex
import tensorflow as tf
from tensorflow.keras import mixed_precision

# ---------- Paths ----------
CUDA_ID  = "0"

# New run root: your v4 experiment directory
RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4")

# New training split: hires-only training set from the 50/25/25 split
TRAIN_DIR   = Path(
    "/home/rbielski/ARC/ds004884/derivatives/aggregates/"
    "t1w_with_masks/mni_1mm_ants_fixed/_standardized/"
    "_resolution_v2/_splits_50_25_25/train_hires"
)
TRAIN_T1    = TRAIN_DIR / "t1"
TRAIN_MASKS = TRAIN_DIR / "masks"

# Training module stays the same
MODULE_PATH = Path(
    "/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py"
)

# ---------- New run folders (won't overwrite old) ----------
RUN_ID        = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR       = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR     = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
LOG_DIR       = RUN_DIR / "logs"
for d in (MODEL_DIR, CALLBACKS_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- Env & precision ----------
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_ID
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["SMARTSOTA_LOG_DIR"] = str(LOG_DIR)

tf.keras.backend.clear_session(); gc.collect()
mixed_precision.set_global_policy("mixed_float16")

# ---------- Tee logs to both notebook and file ----------
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data); s.flush()
        return len(data)
    def flush(self):
        for s in self.streams:
            s.flush()

log_file = open(LOG_DIR / "train_stdout_stderr.log", "a", buffering=1)
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

print("Run ID:", RUN_ID)
print("Run root:", RUN_ROOT)
print("Training dir:", TRAIN_DIR)
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("set_memory_growth failed:", e)

# ---------- Import training module; avoid MirroredStrategy to save VRAM ----------
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
seg.tf = tf  # in case the module references tf early
spec.loader.exec_module(seg)
seg.strategy = tf.distribute.get_strategy()
print("Strategy:", type(seg.strategy).__name__)

# ---------- Hyperparameters (same as v3 unless you want to tweak) ----------
INPUT_SHAPE    = (192, 224, 192, 1)
BATCH_SIZE     = 1
BASE_FILTERS   = 6
SAM_HEADS      = 2
VAL_SPLIT      = 0.15
TOTAL_EPOCHS   = 300
INITIAL_EPOCH  = 0

# Learning-rate schedule knobs
INITIAL_LR     = 1.5e-5
MIN_LR         = 5e-7
WARMUP_EPOCHS  = 5

# Augmentation & small-lesion emphasis
AUG_INTENSITY          = 0.45
ROTATION_RANGE         = 25
SMALL_LESION_THRESHOLD = 6000
SYNTHETIC_LESION_PROB  = 0.60

# Loss mixture
DICE_WEIGHT     = 0.30
BOUNDARY_WEIGHT = 0.70

# Optional regularization
DROPOUT_RATE = 0.60
L2_REG       = 1.0e-3

# ---------- Launch training: absolutely fresh (no resume/no load) ----------
try:
    history = seg.train_dynamic_model(
        # Data roots – pass separated subfolders
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,

        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,

        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,

        INPUT_SHAPE=INPUT_SHAPE,
        BATCH_SIZE=BATCH_SIZE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        RESAMPLE_TO_TARGET=True,

        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        SMALL_LESION_THRESHOLD=SMALL_LESION_THRESHOLD,
        SYNTHETIC_LESION_PROB=SYNTHETIC_LESION_PROB,
        ROTATION_RANGE=ROTATION_RANGE,
        VALIDATION_SPLIT=VAL_SPLIT,

        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,

        DICE_WEIGHT=DICE_WEIGHT,
        BOUNDARY_WEIGHT=BOUNDARY_WEIGHT,
        DROPOUT_RATE=DROPOUT_RATE,
        L2_REG=L2_REG,
    )
    print("Training complete. Logged keys:", list(getattr(history, "history", {}).keys()))
    print("Run dir:", RUN_DIR)

except Exception as e:
    print("\n================= UNCAUGHT EXCEPTION =================")
    traceback.print_exc()
    print("======================================================\n")
    try:
        print("Last few GPU snapshots:")
        for _ in range(2):
            subprocess.run(shlex.split("nvidia-smi"), check=False)
            time.sleep(1)
    except Exception:
        pass
    raise
finally:
    try:
        log_file.flush()
    except Exception:
        pass


2025-11-26 12:12:50.092315: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Run ID: 20251126_121251
Run root: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4
Training dir: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_resolution_v2/_splits_50_25_25/train_hires
TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy: _DefaultDistributionStrategy
Strategy: _DefaultDistributionStrategy


2025-11-26 12:12:51,877 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-26 12:12:51,877 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-26 12:12:51,877 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 1
2025-11-26 12:12:51,880 - SmartSOTA_Dynamic - INFO - 🧭 INPUT_SHAPE set to: (192, 224, 192, 1)
2025-11-26 12:12:51,880 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
I0000 00:00:1764184371.991379 2243210 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1764184371.992465 2243210 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20529 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
2025-11-26 12:12:51,995 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.73GB | GPU mem track

2025-11-26 12:13:47,751 - SmartSOTA_Dynamic - INFO - Model: "SmartSOTA_Dynamic"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 192, 224,  │          0 │ -                 │
│ (InputLayer)        │ 192, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_block │ (None, 192, 224,  │      1,194 │ input_layer[0][0] │
│ (ResidualConvBlock) │ 192, 6)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block  │ (None, 192, 224,  │      4,050 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 192, 6)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

Epoch 1/300


...

166/343 ━━━━━━━━━━━━━━━━━━━━ 46s 264ms/step - dice_coefficient: 0.1405 - loss: 0.3024

2025-11-26 17:07:45,975 - SmartSOTA_Dynamic - INFO - Memory at batch_57790: CPU=13.78GB | GPU mem tracking failed | Disk: 1220.3GB free


175/343 ━━━━━━━━━━━━━━━━━━━━ 43s 261ms/step - dice_coefficient: 0.1396 - loss: 0.3027

2025-11-26 17:07:47,995 - SmartSOTA_Dynamic - INFO - Memory at batch_57800: CPU=13.71GB | GPU mem tracking failed | Disk: 1220.3GB free


183/343 ━━━━━━━━━━━━━━━━━━━━ 41s 258ms/step - dice_coefficient: 0.1389 - loss: 0.3029